## Predicting Floods in Sub-Saharan Africa Using Satellite Imagery and Computer Vision

# Project Objective

Developing a machine learning system that can predict floods 3 days in advance in Sub-Saharan Africa using Satellite imagery and related environmental data. The focus will be on demonstrating the application of computer vision techniques to extract spatial patterns predictive of flooding


# Motivation

- Floods are a major hazard in Sub-Saharan Africa, causing property damage, displacement, and loss of life.
- Early prediction enables disaster preparedness and humanitarian response.
- Satellite imagery is increasingly available, offering a rich visual dataset for flood prediction.
- This project demonstrates the use of computer vision pipelines in a real-world social-impact context.


In [2]:
# Load necessary libraries

# Core
import os
import glob
from pathlib import Path

import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import geemap

# Geospatial + raster
import rasterio
from rasterio.plot import show
import geopandas as gpd
import ee

# ML utilities / metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

# Notebook display
from IPython.display import display

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("Imports loaded.")


Imports loaded.


# Priority Countries for the Analysis are:

1. Mozambique: It is a "hotspot" for tropical cyclones (e.g., Idai, Kenneth, Freddy) and has some of the most extensive satellite-based flood archives in Southern Africa. The Beira and Macomia regions have well-documented SAR-based flood maps that are excellent for training
2. Nigeria: Nigeria experiences consistent, annually recurring floods, particularly in the Niger River Basin and Lagos. It has high-resolution, frequently updated datasets that are ideal for demonstrating CNN-based feature extraction in diverse environments
3. Kenya: Kenya has recently transitioned from severe drought to catastrophic flooding (2024-2025), providing a fresh set of "precursor" data for the 7-to-3-day window. Organizations like Digital Earth Africa offer analysis-ready Sentinel-1 data specifically for this region
4. Zimbabwe: I have local context for Manicaland (Chimanimani/Chipinge), it makes a strong case study for validating model's real-world performance against known mountainous terrain challenges

# Mozambique

In [3]:
# Authenticate and initialize Earth Engine
try:
    ee.Initialize()
    print("Earth Engine already initialized.")
except Exception:
    # "Enter verification code:"
    ee.Authenticate(auth_mode="notebook")
    ee.Initialize()
    print("Earth Engine initialized successfully.")

Earth Engine already initialized.


In [4]:
def get_mozambique_flood_data(aoi, start_date, end_date):
    """Filters Sentinel-1 collection for Mozambique flood events."""
    collection = (ee.ImageCollection('COPERNICUS/S1_GRD')
                  .filterBounds(aoi)
                  .filterDate(start_date, end_date)
                  .filter(ee.Filter.eq('instrumentMode', 'IW')) # Interferometric Wide mode
                  .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
                  .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
                  .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))) # Consistent orbit
    return collection

# Beira, Mozambique (Focus of Cyclone Idai)
beira_coords = [34.83, -19.83]
beira_aoi = ee.Geometry.Point(beira_coords).buffer(20000) # 20km radius

# Pulling precursors for Cyclone Idai (Peak was ~March 15, 2019)
# We pull data from 7 days before to capture the 'saturation' features
s1_collection = get_mozambique_flood_data(beira_aoi, '2019-03-08', '2019-03-16')

print(f"Images Found: {s1_collection.size().getInfo()}")

Images Found: 1


Only 1 image was found as the "revisit time" (how often the satellite passes over the exact same spot) is typically 6 to 12 days. This presents a data shortage challenge to build a time series for a 3-day prediction project and will potentially require expanded date ranges or checking additional orbits

In [5]:
# Get the first (and only) image from example collection
image = s1_collection.first()

# Print metadata to check the exact acquisition time
info = image.getInfo()
print(f"Image ID: {info['id']}")
print(f"Date: {info['properties']['system:time_start']}") 


Image ID: COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20190314T030905_20190314T030930_026328_02F196_2EC3
Date: 1552532945000


The Image ID refers to a Sentinel-1 SAR (radar) satellite image

The Unix Timestamp of the image represents the number of milliseconds that have passed since January 1, 1970 UTC and the conversion to get the image capture date to a human-readable format is below

In [6]:
# Transform the date from milliseconds to a human-readable format
import datetime

timestamp_ms = info['properties']['system:time_start']
date_time = datetime.datetime.fromtimestamp(timestamp_ms / 1000.0, tz=datetime.timezone.utc)

print(f"Readable Date: {date_time.strftime('%Y-%m-%d %H:%M:%S')} UTC")

Readable Date: 2019-03-14 03:09:05 UTC


This is a Sentinel-1 SAR image that captures radar backscatter, where flooded areas appear as dark regions due to low signal return, making it highly suitable for flood detection even under cloud cover.

For intuitive visual inspection in RGB, I calculated a false color composite using VV, VH, and the ratio VV/VH

In [7]:
# Calculate false color composite using VV, VH, and their ratio (VV/VH)

# Create an interactive map (geemap) if it doesn't exist yet
try:
    Map
except NameError:
    Map = geemap.Map()

# Center the map on the AOI (Beira) and add an AOI overlay
Map.centerObject(beira_aoi, 9)
Map.addLayer(beira_aoi, {}, "Beira AOI")

# 1. Calculate the ratio (VV/VH)
ratio = image.select('VV').divide(image.select('VH')).rename('ratio')

# 2. Add the ratio as a new band to the image
rgb_image = image.addBands(ratio)

# 3. Set visualization parameters (adjust min/max for specific contrast)
# Typically, dB values for VV/VH range from -25 to 0
vis_params = {
    'bands': ['VV', 'VH', 'ratio'],
    'min': [-25, -25, 0.5],
    'max': [0, -5, 2.0]
}

# 4. Add to map using geemap
Map.addLayer(rgb_image, vis_params, 'Sentinel-1 False Color RGB')

# Display the map in the notebook
Map


Map(center=[-19.82997948235411, 34.83000060484723], controls=(WidgetControl(options=['position', 'transparent_…

As can be observed from the image, water areas are much darker as radar waves scatter when they hit the smooth water compared to hard land

Furthermore, Beira is next to the Indian Ocean which would appear "flooded" and potentially distort the analysis

## Option 2: Z